<a href="https://colab.research.google.com/github/andadivaan-spec/MeTernakFINAL/blob/main/YOLOv8(LR0.03BT8).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# YOLO LENDIR SAPI — FIXED SCRIPT (Anti-Overfitting + Min 50% Accuracy)
# Jalankan di Google Colab dengan GPU
# ============================================================

!pip install -q ultralytics albumentations

import os, cv2, shutil, random
import numpy as np
import albumentations as A
from google.colab import files
from ultralytics import YOLO
import zipfile

# ── 0. UPLOAD & EXTRACT ──────────────────────────────────────────────────────
uploaded = files.upload()
for fname in uploaded.keys():
    with zipfile.ZipFile(f"/content/{fname}") as zf:
        zf.extractall("/content/extracted")

# ── 1. POLYGON → BBOX ────────────────────────────────────────────────────────
def polygon_to_bbox(lbl_path):
    with open(lbl_path) as f:
        lines = f.read().strip().splitlines()
    new_lines = []
    for line in lines:
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        coords = list(map(float, parts[1:]))
        if len(coords) == 4:
            new_lines.append(line)
            continue
        xs = coords[0::2]; ys = coords[1::2]
        cx = (min(xs) + max(xs)) / 2
        cy = (min(ys) + max(ys)) / 2
        w  = max(xs) - min(xs)
        h  = max(ys) - min(ys)
        # Clamp to [0,1]
        cx = min(max(cx, 0), 1); cy = min(max(cy, 0), 1)
        w  = min(max(w,  0), 1); h  = min(max(h,  0), 1)
        if w > 0.01 and h > 0.01:  # buang bbox terlalu kecil
            new_lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    with open(lbl_path, "w") as f:
        f.write("\n".join(new_lines))

for split in ["train", "valid", "test"]:
    lbl_dir = f"/content/extracted/{split}/labels"
    if not os.path.exists(lbl_dir): continue
    for fname in os.listdir(lbl_dir):
        if fname.endswith(".txt"):
            polygon_to_bbox(f"{lbl_dir}/{fname}")
print("Polygon → bbox selesai")

# ── 2. SETUP DIRS ─────────────────────────────────────────────────────────────
IMG_DIR = "/content/extracted/train/images"
LBL_DIR = "/content/extracted/train/labels"
OUT_IMG = "/content/dataset/images/train"
OUT_LBL = "/content/dataset/labels/train"
os.makedirs(OUT_IMG, exist_ok=True)
os.makedirs(OUT_LBL, exist_ok=True)

# Kumpulkan seed per kelas
seeds = {0: [], 1: [], 2: []}
for fname in os.listdir(IMG_DIR):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")): continue
    lbl = f"{LBL_DIR}/{fname.rsplit('.',1)[0]}.txt"
    if not os.path.exists(lbl): continue
    with open(lbl) as f:
        lines = f.read().strip().splitlines()
    if not lines: continue
    cls = int(lines[0].split()[0])
    if cls in seeds:
        seeds[cls].append(fname)
        shutil.copy(f"{IMG_DIR}/{fname}", f"{OUT_IMG}/{fname}")
        shutil.copy(lbl, f"{OUT_LBL}/{fname.rsplit('.',1)[0]}.txt")

for cls, fnames in seeds.items():
    print(f"Kelas {cls}: {len(fnames)} gambar seed")

# ── 3. AUGMENTASI AGRESIF (TARGET 300/KELAS) ─────────────────────────────────
# Target tinggi agar model tidak overfit ke 3-4 gambar per kelas
TARGET_PER_CLASS = 300

aug = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.7),
    A.Rotate(limit=30, p=0.6),
    A.GaussianBlur(blur_limit=(3, 9), p=0.4),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=30, p=0.6),
    A.RandomScale(scale_limit=0.3, p=0.4),
    A.CLAHE(clip_limit=4.0, p=0.4),
    A.RandomShadow(p=0.3),
    A.GaussNoise(p=0.3),
    A.Perspective(scale=(0.05, 0.15), p=0.4),
    A.Resize(640, 640),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

for cls, fnames in seeds.items():
    if not fnames: continue
    count, attempts = 0, 0
    while count < TARGET_PER_CLASS and attempts < TARGET_PER_CLASS * 20:
        attempts += 1
        fname = random.choice(fnames)
        stem  = fname.rsplit(".", 1)[0]
        img   = cv2.imread(f"{IMG_DIR}/{fname}")
        if img is None: continue

        bboxes, class_labels = [], []
        with open(f"{LBL_DIR}/{stem}.txt") as f:
            for line in f.read().strip().splitlines():
                p = line.split()
                if len(p) == 5:
                    class_labels.append(int(p[0]))
                    bboxes.append([float(p[1]), float(p[2]), float(p[3]), float(p[4])])
        if not bboxes: continue

        try:
            res = aug(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
                      bboxes=bboxes, class_labels=class_labels)
            if not res["bboxes"]: continue
            new_stem = f"c{cls}_aug_{count:05d}"
            cv2.imwrite(f"{OUT_IMG}/{new_stem}.jpg",
                        cv2.cvtColor(res["image"], cv2.COLOR_RGB2BGR))
            with open(f"{OUT_LBL}/{new_stem}.txt", "w") as lf:
                for c2, bb in zip(res["class_labels"], res["bboxes"]):
                    lf.write(f"{c2} {bb[0]:.6f} {bb[1]:.6f} {bb[2]:.6f} {bb[3]:.6f}\n")
            count += 1
        except:
            continue
    print(f"Kelas {cls}: {count} gambar augmentasi")

print(f"Total train: {len(os.listdir(OUT_IMG))} gambar")

# ── 4. COPY VALID & TEST ──────────────────────────────────────────────────────
for split in ["valid", "test"]:
    for sub in ["images", "labels"]:
        src = f"/content/extracted/{split}/{sub}"
        dst = f"/content/dataset/{sub}/{split}"
        if os.path.exists(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)

# ── 5. YAML ───────────────────────────────────────────────────────────────────
with open("/content/dataset.yaml", "w") as f:
    f.write(
        "path: /content/dataset\n"
        "train: images/train\n"
        "val: images/valid\n"
        "test: images/test\n"
        "nc: 3\n"
        "names:\n"
        "  0: lendir_kuning_keruh\n"
        "  1: lendir_transparan\n"
        "  2: lendir_darah\n"
    )

# ── 6. TRAIN ──────────────────────────────────────────────────────────────────
# Pakai yolov8n (nano) — lebih ringan, lebih sedikit overfit untuk dataset kecil
model = YOLO("yolov8n.pt")

model.train(
    data          = "/content/dataset.yaml",
    epochs        = 100,           # Lebih banyak epoch
    imgsz         = 640,
    batch         = 8,
    device        ="cpu",
    patience      = 20,            # Early stopping lebih sabar
    cos_lr        = True,
    warmup_epochs = 5,
    lr0           = 0.03,          # LR lebih rendah agar stabil
    lrf           = 0.001,
    weight_decay  = 0.001,
    dropout       = 0.1,           # Dropout anti-overfit
    augment       = True,
    mixup         = 0.1,           # Mixup augmentasi
    copy_paste    = 0.1,
    mosaic        = 1.0,
    cache         = True,
    workers       = 2,
    project       = "/content/runs",
    name          = "lendir_fixed",
    save          = True,
    plots         = True,
    val           = True,
)

# ── 7. VALIDASI ───────────────────────────────────────────────────────────────
metrics = model.val()
print(f"\n── HASIL ──")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

# ── 8. EXPORT KE ONNX ────────────────────────────────────────────────────────
model.export(format="onnx", imgsz=640, opset=12, simplify=True)
print("\nFile ONNX tersimpan, download dari:")
print("/content/runs/lendir_fixed/weights/best.onnx")

# Download otomatis
files.download("/content/runs/lendir_fixed/weights/best.onnx")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Saving ivdaAndaanuptri.v1i.yolov8.zip to ivdaAndaanuptri.v1i.yolov8.zip
Polygon → bbox selesai
Kelas 0: 4 gambar seed
Kelas 1: 3 gambar seed
Kelas 2: 3 gambar seed
Kelas 0: 300 gambar augmentasi
Kelas 1: 300 gambar augmentasi
Kelas 2: 300 gambar augmentasi
Total train: 910 gambar
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>